In [ ]:
import pandas as pd
import numpy as np

from typing import List

import re

from collections import defaultdict
import glob
import seaborn as sns
import matplotlib.pyplot as plt
import json
from typing import List, Any, Set, Dict
from tqdm import tqdm
import re
sns.set()
import math


## Load all reports

In [35]:
import json
import glob
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm  # Make sure to install this: !pip install tqdm

def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

if __name__ == '__main__':
    file_pattern = "data/*.json"
    files = glob.glob(file_pattern)
    
    data_list = []
    
    # We use a context manager for the executor
    with ProcessPoolExecutor() as executor:
        # map() returns an iterator; we wrap it with tqdm to get the progress bar
        # total=len(files) helps tqdm estimate the time remaining
        report_folder = 'outputs/reports/offline_wcd_igraph'
        report_files = glob.glob(report_folder + "/*.json")
        results = executor.map(load_json, report_files)
        
        # This loop pulls results from the pool as they finish
        for result in tqdm(results, total=len(report_files), desc="Loading JSONs"):
            data_list.append(result)
            
    print(f"Finished loading {len(data_list)} files.")

Loading JSONs: 100%|██████████| 4176/4176 [01:12<00:00, 57.72it/s] 


Finished loading 4176 files.


## Find all legacy reports with no system prompt

In [53]:
prompt_file = "wildguard_prompts_larger.parquet"
prompts_df = pd.read_parquet(prompt_file)
prompts = prompts_df['prompt'].tolist()
harmful_prompts = prompts[:30]
non_harmful_prompts = prompts[30:]


report_folder = 'outputs/reports/offline_wcd_igraph'
report_files = glob.glob(report_folder + "/*.json")

def _slugify_for_filename(value: str, max_len: int = 80) -> str:
    compact = re.sub(r"\s+", "_", value.strip().lower())
    safe = re.sub(r"[^a-z0-9._=-]", "-", compact)
    safe = re.sub(r"-+", "-", safe).strip("-_.")
    if not safe:
        return "na"
    return safe[:max_len]

# agg params
temperature = 0.4
top_p = 0.8

def agg_stats(prompt_list: List[str], temp: float, top_p: float):
    file_found = 0
    file_confirmed = 0
    wcds = defaultdict(list)
    pruned_masses = defaultdict(list)

    for prompt in tqdm(prompt_list, desc="Processing prompts"):
        # find the file corresponding to the prompt
        for report in data_list:
            # if str(temperature).replace('.', 'p') in report_file and str(top_p).replace('.', 'p') in report_file and _slugify_for_filename(prompt)[:20] in report_file:
            #     file_found += 1
            if True:
                if re.sub(r'[^a-zA-Z0-9]', '', report["prompt"].strip()) == re.sub(r'[^a-zA-Z0-9]', '', prompt.strip()) and float(report["temperature"]) == temp and float(report["top_p"]) == top_p and "system_prompt" not in report:  # match
                    file_confirmed += 1
                    intervention = report["interventions"][0]

                    if intervention["intervention"] == "none":
                        intervention_slug = "none"
                    else:
                        intervention_slug = str(intervention["intervention"]) + "_" + str(intervention["k"]) + "_" + str(intervention['selection']) + "_" + str(intervention['top_n'])

                    wcds[intervention_slug].append(intervention["wcd"])
                    pruned_masses[intervention_slug].append(intervention["mass_nucleus_after_intervention"])
    print(file_found, file_confirmed)
    return wcds, pruned_masses

harmful_wcds, harmful_pruned_masses = agg_stats(harmful_prompts, temperature, top_p)
non_harmful_wcds, non_harmful_pruned_masses = agg_stats(non_harmful_prompts, temperature, top_p)

Processing prompts: 100%|██████████| 30/30 [00:10<00:00,  2.78it/s]


0 870


Processing prompts: 100%|██████████| 30/30 [00:06<00:00,  4.29it/s]

0 870


## Find all new reports with system prompt

In [54]:
def _format_float_for_filename(value: float) -> str:
    formatted = f"{value:g}"
    return formatted.replace("-", "m").replace(".", "p")


def _extract_prompt_slug(text: str) -> str:
    if not text:
        return ""
    marker_start = "prompt="
    marker_end = "__model="
    start_idx = text.find(marker_start)
    if start_idx == -1:
        return ""
    start_idx += len(marker_start)
    end_idx = text.find(marker_end, start_idx)
    if end_idx == -1:
        return ""
    return text[start_idx:end_idx]


def _build_safety_intervention_slug(intervention: Dict[str, Any]) -> str:
    if intervention.get("intervention") == "none":
        return "safety_system_prompt__none"
    return (
        "safety_system_prompt__"
        f"{intervention.get('intervention')}_{intervention.get('k')}_"
        f"{intervention.get('selection')}_{intervention.get('top_n')}"
    )


def _prefilter_safety_report_files(paths: List[str], temp: float, top_p: float) -> List[str]:
    temp_token = f"__temp={_format_float_for_filename(temp)}"
    top_p_token = f"__top_p={_format_float_for_filename(top_p)}"
    system_prompt_token = "__system_prompt=safety_system_prompt"

    filtered = []
    for path in paths:
        name = path.rsplit("/", 1)[-1]
        if temp_token not in name or top_p_token not in name:
            continue
        if system_prompt_token not in name:
            continue
        filtered.append(path)
    return filtered


def _load_safety_report_index(paths: List[str]) -> List[Dict[str, Any]]:
    indexed_reports = []
    for report_file in tqdm(paths, desc="Indexing safety-system reports"):
        try:
            with open(report_file, "r", encoding="utf-8") as f:
                report = json.load(f)
        except (OSError, json.JSONDecodeError):
            continue

        if str(report.get("system_prompt_id", "")).strip() != "safety_system_prompt":
            continue
        interventions = report.get("interventions") or []
        if not interventions:
            continue
        try:
            temperature_value = float(report.get("temperature"))
            top_p_value = float(report.get("top_p"))
        except (TypeError, ValueError):
            continue

        jsonl_path = str(report.get("jsonl", ""))
        prompt_slug = _extract_prompt_slug(jsonl_path)
        if not prompt_slug:
            prompt_slug = _extract_prompt_slug(report_file)

        indexed_reports.append(
            {
                "report_file": report_file,
                "prompt_slug": prompt_slug,
                "temperature": temperature_value,
                "top_p": top_p_value,
                "intervention": interventions[0],
            }
        )

    return indexed_reports


safety_candidate_report_files = _prefilter_safety_report_files(
    report_files,
    temp=temperature,
    top_p=top_p,
)
print(f"Safety-system report candidates: {len(safety_candidate_report_files)} / {len(report_files)}")

safety_indexed_reports = _load_safety_report_index(safety_candidate_report_files)
print(f"Safety-system indexed reports: {len(safety_indexed_reports)}")


def agg_safety_stats(prompt_list: List[str], temp: float, top_p: float):
    wcds = defaultdict(list)
    pruned_masses = defaultdict(list)
    target_prompt_slugs = {_slugify_for_filename(p) for p in prompt_list}

    for report in tqdm(safety_indexed_reports, desc="Aggregating safety-system reports"):
        if report["prompt_slug"] not in target_prompt_slugs:
            continue
        if not math.isclose(float(report["temperature"]), float(temp), abs_tol=1e-12):
            continue
        if not math.isclose(float(report["top_p"]), float(top_p), abs_tol=1e-12):
            continue

        intervention = report["intervention"]
        intervention_slug = _build_safety_intervention_slug(intervention)
        wcds[intervention_slug].append(intervention.get("wcd"))
        pruned_masses[intervention_slug].append(intervention.get("mass_nucleus_after_intervention"))

    return wcds, pruned_masses


safety_harmful_wcds, safety_harmful_pruned_masses = agg_safety_stats(harmful_prompts, temperature, top_p)
safety_non_harmful_wcds, safety_non_harmful_pruned_masses = agg_safety_stats(non_harmful_prompts, temperature, top_p)

Safety-system report candidates: 1740 / 4176


Indexing safety-system reports:   0%|          | 0/1740 [00:00<?, ?it/s]

Indexing safety-system reports: 100%|██████████| 1740/1740 [00:40<00:00, 43.38it/s] 


Safety-system indexed reports: 1740


Aggregating safety-system reports: 100%|██████████| 1740/1740 [00:00<00:00, 113282.15it/s]


## Transform to DF

In [74]:
import pandas as pd
harmful_wcds_df = pd.DataFrame(harmful_wcds)
non_harmful_wcds_df = pd.DataFrame(non_harmful_wcds)
harmful_pruned_masses_df = pd.DataFrame(harmful_pruned_masses)
non_harmful_pruned_masses_df = pd.DataFrame(non_harmful_pruned_masses)

cols = [
    col
    for col in harmful_wcds_df.columns
    if "30" not in col and "40" not in col and "random" not in col
]
harmful_wcds_df = harmful_wcds_df / harmful_wcds_df["none"].values[:, None]
non_harmful_wcds_df = non_harmful_wcds_df / non_harmful_wcds_df["none"].values[:, None]
harmful_pruned_masses_df = harmful_pruned_masses_df / harmful_pruned_masses_df["none"].values[:, None]
non_harmful_pruned_masses_df = non_harmful_pruned_masses_df / non_harmful_pruned_masses_df["none"].values[:, None]

harmful_wcds_df = harmful_wcds_df[cols]
non_harmful_wcds_df = non_harmful_wcds_df[cols]
harmful_pruned_masses_df = harmful_pruned_masses_df[cols]
non_harmful_pruned_masses_df = non_harmful_pruned_masses_df[cols]
combined_wcds_df = pd.concat([harmful_wcds_df, non_harmful_wcds_df], axis=0)
combined_pruned_masses_df = pd.concat([harmful_pruned_masses_df, non_harmful_pruned_masses_df], axis=0)

In [79]:
combined_wcds_df.shape

(60, 11)

In [75]:
import pandas as pd
safety_harmful_wcds_df = pd.DataFrame(safety_harmful_wcds)
safety_non_harmful_wcds_df = pd.DataFrame(safety_non_harmful_wcds)
safety_harmful_pruned_masses_df = pd.DataFrame(safety_harmful_pruned_masses)
safety_non_harmful_pruned_masses_df = pd.DataFrame(safety_non_harmful_pruned_masses)

cols = [
    col
    for col in safety_harmful_wcds_df.columns
    if "30" not in col and "40" not in col and "random" not in col
]
safety_harmful_wcds_df = safety_harmful_wcds_df / safety_harmful_wcds_df["safety_system_prompt__none"].values[:, None]
safety_non_harmful_wcds_df = safety_non_harmful_wcds_df / safety_non_harmful_wcds_df["safety_system_prompt__none"].values[:, None]
safety_harmful_pruned_masses_df = safety_harmful_pruned_masses_df / safety_harmful_pruned_masses_df["safety_system_prompt__none"].values[:, None]
safety_non_harmful_pruned_masses_df = safety_non_harmful_pruned_masses_df / safety_non_harmful_pruned_masses_df["safety_system_prompt__none"].values[:, None]

safety_harmful_wcds_df = safety_harmful_wcds_df[cols]
safety_non_harmful_wcds_df = safety_non_harmful_wcds_df[cols]
safety_harmful_pruned_masses_df = safety_harmful_pruned_masses_df[cols]
safety_non_harmful_pruned_masses_df = safety_non_harmful_pruned_masses_df[cols]
combined_safety_wcds_df = pd.concat([safety_harmful_wcds_df, safety_non_harmful_wcds_df], axis=0)
combined_safety_pruned_masses_df = pd.concat([safety_harmful_pruned_masses_df, safety_non_harmful_pruned_masses_df], axis=0)

In [76]:
combined_safety_wcds_df.shape

(60, 11)

## Get aggregates for overleaf

In [82]:
## success rate
for col in combined_wcds_df.columns:
    print(col, "success rate:", round((combined_wcds_df[col] < 1).mean(), 2))

fixed_k_10_extreme_3 success rate: 0.016666666666666666
fixed_k_20_min_1 success rate: 0.13333333333333333
fixed_k_10_max_1 success rate: 0.16666666666666666
fixed_k_10_extreme_1 success rate: 0.16666666666666666
fixed_k_20_extreme_3 success rate: 0.0
fixed_k_20_max_1 success rate: 0.11666666666666667
fixed_k_20_both_sides_1 success rate: 0.0
fixed_k_10_min_1 success rate: 0.15
fixed_k_10_both_sides_1 success rate: 0.06666666666666667
fixed_k_20_extreme_1 success rate: 0.15
none success rate: 0.0


In [ ]:
def summarize_and_plot_wcd_vs_mass(
    wcd_df: pd.DataFrame,
    pruned_mass_df: pd.DataFrame,
    group_name: str = "group",
    include_none: bool = False
) -> pd.DataFrame:
    # Keep only shared columns
    shared_cols = [c for c in wcd_df.columns if c in pruned_mass_df.columns]
    if not include_none:
        shared_cols = [c for c in shared_cols if c != "none"]

    summary_df = pd.DataFrame({
        "column": shared_cols,
        "avg_wcd": [wcd_df[c].mean() for c in shared_cols],
        "avg_pruned_mass": [pruned_mass_df[c].mean() for c in shared_cols],
    }).sort_values("avg_wcd", ascending=False).reset_index(drop=True)

    display(summary_df)

    plt.figure(figsize=(9, 6))
    sns.scatterplot(data=summary_df, x="avg_wcd", y="avg_pruned_mass", s=90)
    for _, r in summary_df.iterrows():
        plt.text(r["avg_wcd"], r["avg_pruned_mass"], r["column"], fontsize=8, alpha=0.8)

    plt.title(f"Average WCD vs Average Pruned Mass ({group_name})")
    plt.xlabel("Average WCD")
    plt.ylabel("Average Pruned Mass")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    return summary_df


# Harmful
harmful_summary = summarize_and_plot_wcd_vs_mass(
    harmful_wcds_df,
    harmful_pruned_masses_df,
    group_name="harmful"
)

# Non-harmful
non_harmful_summary = summarize_and_plot_wcd_vs_mass(
    non_harmful_wcds_df,
    non_harmful_pruned_masses_df,
    group_name="non-harmful"
)

# combined
combined_summary = summarize_and_plot_wcd_vs_mass(
    pd.concat([harmful_wcds_df, non_harmful_wcds_df], ignore_index=True, axis=0),
    pd.concat([harmful_pruned_masses_df, non_harmful_pruned_masses_df], ignore_index=True, axis=0),
    group_name="combined"
)

In [ ]:
def summarize_ratio_lt1(
    ratio_df: pd.DataFrame,
    group_name: str = "group",
    include_none: bool = False,
    plot: bool = True,
):
    cols = ratio_df.columns.tolist()
    if not include_none and "none" in cols:
        cols = [c for c in cols if c != "none"]

    ratio_dist_by_col = {}
    total_n = len(ratio_df)

    for col in cols:
        vals = ratio_df.loc[ratio_df[col] < 1, col].dropna().astype(float)
        ratio_dist_by_col[col] = vals.reset_index(drop=True)

    summary_df = pd.DataFrame([
        {
            "column": col,
            "n_ratio_lt_1": len(vals),
            "pct_ratio_lt_1": (len(vals) / total_n) if total_n > 0 else np.nan,
            "mean_ratio": vals.mean() if len(vals) else np.nan,
            "std_ratio": vals.std() if len(vals) else np.nan,
            "min_ratio": vals.min() if len(vals) else np.nan,
            "q25_ratio": vals.quantile(0.25) if len(vals) else np.nan,
            "median_ratio": vals.median() if len(vals) else np.nan,
            "q75_ratio": vals.quantile(0.75) if len(vals) else np.nan,
            "max_ratio": vals.max() if len(vals) else np.nan,
        }
        for col, vals in ratio_dist_by_col.items()
    ]).sort_values("n_ratio_lt_1", ascending=False).reset_index(drop=True)

    display(summary_df)

    long_df = pd.concat(
        [pd.DataFrame({"column": col, "ratio": vals}) for col, vals in ratio_dist_by_col.items()],
        ignore_index=True
    ) if len(ratio_dist_by_col) else pd.DataFrame(columns=["column", "ratio"])

    if plot and not long_df.empty:
        plt.figure(figsize=(12, 5))
        sns.boxplot(data=long_df, x="column", y="ratio")
        plt.xticks(rotation=60, ha="right")
        plt.title(f"Ratio distribution where ratio < 1 ({group_name})")
        plt.tight_layout()
        plt.show()

    return summary_df, long_df, ratio_dist_by_col


# Usage
non_harmful_summary_lt1, non_harmful_long_lt1, non_harmful_dist_lt1 = summarize_ratio_lt1(
    ratio_df=non_harmful_wcds_df,
    group_name="non-harmful"
)

harmful_summary_lt1, harmful_long_lt1, harmful_dist_lt1 = summarize_ratio_lt1(
    ratio_df=harmful_wcds_df,
    group_name="harmful"
)

summarize_ratio_lt1(
    ratio_df=pd.concat([harmful_wcds_df, non_harmful_wcds_df], ignore_index=True, axis=0),
    group_name="combined"
)


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

# ---------------------------
# Combine harmful + non-harmful
# ---------------------------
combined_wcds_df = pd.concat(
    [harmful_wcds_df, non_harmful_wcds_df],
    axis=0,
    ignore_index=True
)

combined_pruned_masses_df = pd.concat(
    [harmful_pruned_masses_df, non_harmful_pruned_masses_df],
    axis=0,
    ignore_index=True
)

# Make sure values are numeric
combined_wcds_df = combined_wcds_df.apply(pd.to_numeric, errors="coerce")
combined_pruned_masses_df = combined_pruned_masses_df.apply(pd.to_numeric, errors="coerce")


def one_sided_wilcoxon_vs_one(
    df: pd.DataFrame,
    correction_method: str = "holm",
    alternative: str = "less",
):
    """
    For each intervention column (except 'none') in a normalized dataframe,
    test whether values are significantly LESS than 1 using a one-sided
    Wilcoxon signed-rank test.

    H0: median(x - 1) == 0
    H1: median(x - 1) < 0
    """
    intervention_cols = [c for c in df.columns if c != "none"]
    results = []

    for col in intervention_cols:
        x = df[col].dropna().astype(float)
        diffs = x - 1.0
        nonzero_diffs = diffs[diffs != 0]

        if len(x) == 0:
            results.append({
                "intervention": col,
                "n": 0,
                "mean_ratio": np.nan,
                "median_ratio": np.nan,
                "mean_diff_from_1": np.nan,
                "median_diff_from_1": np.nan,
                "wilcoxon_stat": np.nan,
                "p_value": np.nan,
            })
            continue

        if len(nonzero_diffs) == 0:
            # all values are exactly 1
            results.append({
                "intervention": col,
                "n": len(x),
                "mean_ratio": x.mean(),
                "median_ratio": x.median(),
                "mean_diff_from_1": diffs.mean(),
                "median_diff_from_1": diffs.median(),
                "wilcoxon_stat": 0.0,
                "p_value": 1.0,
            })
            continue

        stat, p = wilcoxon(
            x=diffs,
            zero_method="wilcox",
            alternative=alternative,   # "less"
            correction=False,
            mode="auto",
        )

        results.append({
            "intervention": col,
            "n": len(x),
            "mean_ratio": x.mean(),
            "median_ratio": x.median(),
            "mean_diff_from_1": diffs.mean(),
            "median_diff_from_1": diffs.median(),
            "wilcoxon_stat": stat,
            "p_value": p,
        })

    results_df = pd.DataFrame(results)

    valid = results_df["p_value"].notna()
    reject, p_corr, _, _ = multipletests(
        results_df.loc[valid, "p_value"],
        method=correction_method
    )

    results_df.loc[valid, "p_value_holm"] = p_corr
    results_df.loc[valid, "reject_null"] = reject

    return results_df.sort_values(["p_value_holm", "p_value"], na_position="last").reset_index(drop=True)


# ---------------------------
# Run tests
# ---------------------------
wcd_results = one_sided_wilcoxon_vs_one(
    combined_wcds_df,
    correction_method="holm",
    alternative="less",
)

pruned_mass_results = one_sided_wilcoxon_vs_one(
    combined_pruned_masses_df,
    correction_method="holm",
    alternative="less",
)

print("WCD one-sided Wilcoxon results (H1: ratio < 1)")
display(wcd_results)

print("Pruned mass one-sided Wilcoxon results (H1: ratio < 1)")
display(pruned_mass_results)